In [ ]:
import os
os.chdir('f:\\Projects\\GeneIndex')

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
import pandas as pd
import numpy as np
import json
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision import tv_tensors
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection import FasterRCNN
from torchvision.ops import MultiScaleRoIAlign

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
import pickle as pkl
from safetensors.torch import load_file

from torch.utils.tensorboard import SummaryWriter

In [ ]:
from src.m0_ae.encoder import MAEEncoder

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
def read_labels(images_path:Path, labels_file_path:Path|None=None):
    if labels_file_path is None:
        labels_file_path = images_path / '0_labels.json'

    labs_json = json.load(labels_file_path.open())

    labs = []
    imgs = []   

    for k,v in labs_json.items():
        if len(v['regions']) == 0:
            continue
        imgs.append(images_path / v['filename'])
        l = []

        try:
            for rec in v['regions']:
                assert rec['shape_attributes']['name'] == 'rect'
                
                assert rec['shape_attributes']['x'] >0
                assert rec['shape_attributes']['y'] >0
                assert rec['shape_attributes']['width'] >0
                assert rec['shape_attributes']['height'] >0

                # assert rec['region_attributes'].get('act_num')
                l.append([rec['shape_attributes']['x'],
                        rec['shape_attributes']['y'],
                        rec['shape_attributes']['width'],
                        rec['shape_attributes']['height'],
                        int(rec['region_attributes'].get('act_num')) if rec['region_attributes'].get('act_num') else None])
        except Exception as e:
            raise Exception(k)
        labs.append(l)
    return labs,imgs

In [ ]:
labs,imgs = read_labels(Path('src/m1_record_splitter/dataset/ds1_births'))
a,b = read_labels(Path('src/m1_record_splitter/dataset/ds3_al1'))
labs.extend(a)
imgs.extend(b)

imgs,labs = pd.Series(imgs), pd.Series(labs)
# imgs = pd.Series(imgs)
# labs = pd.Series(labs)

In [ ]:
# datasets_df
img_prefixes = imgs.map(lambda x:x.name).str.split('-').str[0]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=67)
split_idx_train, split_idx_val = next(gss.split(imgs, labs, img_prefixes))

In [ ]:
train_imgs, train_labs = pd.Series(imgs)[split_idx_train], pd.Series(labs)[split_idx_train]
val_imgs, val_labs = pd.Series(imgs)[split_idx_val], pd.Series(labs)[split_idx_val]

In [ ]:
print('Subgroups in train:', train_imgs.map(lambda x:x.name).str.split('-').str[0].unique())
print('Subgroups in val:', val_imgs.map(lambda x:x.name).str.split('-').str[0].unique())

In [ ]:
IMG_W = 1920
IMG_H = 1600

image_pre_trans = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Resize([IMG_H,IMG_W]),
])

class ScanDataset(Dataset):
    def __init__(self,image_paths,labels): 
        self.labels = []

        self.images = []

        for image_path,labs in tqdm(list(zip(image_paths,labels)), desc='Loading images to ram'):
            img = Image.open(image_path)
            img.load() # Do not lazy-load
            iw,ih = img.size

            scale_x = IMG_W/iw
            scale_y = IMG_H/ih
            labs = [[x,y,x+w,y+h,act] for [x,y,w,h,act] in labs]
            labs = [[x1*scale_x,y1*scale_y,x2*scale_x,y2*scale_y,act] for [x1,y1,x2,y2,act] in labs]

            img_t = image_pre_trans(img)
            self.images.append(img_t)
            self.labels.append(labs)
        

    def __getitem__(self,idx):
        return tv_tensors.Image(self.images[idx]), {
            "boxes": tv_tensors.BoundingBoxes([i[0:4] for i in self.labels[idx]], format="XYXY", canvas_size=(IMG_H,IMG_W)),
            "labels": torch.tensor([1 if i[-1] is None else 2 for i in self.labels[idx]])
        }
    def __len__(self):
        return len(self.images)

In [ ]:
len(train_imgs)

In [ ]:
train_ds = ScanDataset(train_imgs,train_labs)
val_ds = ScanDataset(val_imgs,val_labs)

In [ ]:
import matplotlib.patches as patches

def show_bbox(img, targs,targs2=None,console=True,ret_fig=False):

    fig, ax = plt.subplots()
    ax.imshow(img.permute(1,2,0))
    
    for [x1,y1,x2,y2], lab in zip(targs['boxes'].tolist(),targs['labels'].tolist()):
        if lab==1:
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
        else:
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='g', facecolor='none')
            ax.add_patch(rect)
    if targs2 is not None:
        for [x1,y1,x2,y2], lab in zip(targs2['boxes'].tolist(),targs2['labels'].tolist()):
            if lab==1:
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='b', facecolor='none')
                ax.add_patch(rect)
            else:
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1, edgecolor='b', facecolor='none')
                ax.add_patch(rect)
    
    if console:
        plt.show()

    if ret_fig:
        return fig
        
    
# tb_writer.add_figure('Pred on val', show_bbox(*train_ds[5], ret_fig=True),global_step=69,close=True)

In [ ]:
D_MODEL = 384
ENCODER_PATH = Path('src/m0_ae/ae-v0.1.safetensors')

class LayerNorm2d(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.ln = nn.LayerNorm(num_channels, eps=eps)
    def forward(self, x):  # x: [N,C,H,W]
        x = x.permute(0, 2, 3, 1)
        x = self.ln(x)
        return x.permute(0, 3, 1, 2)

def out_conv_pair(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 1, bias=False), LayerNorm2d(out_ch),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), LayerNorm2d(out_ch),
    )

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = MAEEncoder(d_model=384)
        self.encoder.load_state_dict(load_file(ENCODER_PATH))
        self.encoder.update_posec_interp()
        print('Loaded backbone encoder from:', ENCODER_PATH)

        d = D_MODEL

        self.down = nn.Sequential(nn.MaxPool2d(2, 2), out_conv_pair(d, d))
        self.up1  = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(d,d//2,3,1,1),
                                   out_conv_pair(d//2, d))
        self.up2  = nn.Sequential(
            nn.Upsample(scale_factor=2), nn.Conv2d(d,d//2,3,1,1), LayerNorm2d(d // 2), nn.GELU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(d//2,d//4,3,1,1),
            out_conv_pair(d//4, d))
        self.feat_conv = out_conv_pair(d, d)

        self.out_channels = d

    def forward(self, x):
        feats = self.encoder(x,mask_perc=0) # [N, 384, x, y]

        down = self.down(feats)
        up1 = self.up1(feats)
        up2 = self.up2(feats)
        feats_conv = self.feat_conv(feats)

        return {
            '0': up2,
            '1': up1,
            '2': feats_conv,
            '3': down
        }

In [ ]:
class BBModel(nn.Module):
    def __init__(self):
        super(BBModel, self).__init__()
        
        anchor_generator = AnchorGenerator(
            sizes=(
                (32,),
                (64,),
                (128,),
                (256,),
            ),
            aspect_ratios=(
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
                (0.5, 1.0, 2.0),
            ),
        )

        roi_pooler = MultiScaleRoIAlign(
            featmap_names=["0", "1", "2", "3"],
            output_size=7,
            sampling_ratio=2,
        )

        self.backbone = Backbone()

        self.model = FasterRCNN(
            backbone=self.backbone,
            num_classes=3,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=1600,
            max_size=1920
        )


    def forward(self, x,targ=None):
        y = self.model(x,targ)

        return y


In [ ]:
import torch

def filter_rcnn_targets(targets, min_size=1.0):
    filtered_targets = []
    
    for target in targets:
        boxes = target.get('boxes')
        
        if boxes is None or len(boxes) == 0:
            filtered_targets.append(target)
            continue
            
        ws = boxes[:, 2] - boxes[:, 0]
        hs = boxes[:, 3] - boxes[:, 1]
        
        valid_mask = (ws > min_size) & (hs > min_size)
        
        if valid_mask.all():
            filtered_targets.append(target)
            continue
            
        filtered_target = {}
        num_boxes = len(boxes)
        
        for key, value in target.items():
            if isinstance(value, torch.Tensor) and len(value) == num_boxes:
                filtered_target[key] = value[valid_mask]
            else:
                filtered_target[key] = value
                
        filtered_targets.append(filtered_target)
        
    return filtered_targets

In [ ]:
aug = v2.Compose([
    # v2.RandomRotation(0),
    # v2.RandomPerspective(distortion_scale=0.3),
    # v2.RandomAffine(10,(0.05,0.05)),
    v2.ColorJitter(0,0,0),
    # v2.ClampBoundingBoxes(),
])

In [ ]:
tb_writer = SummaryWriter('tensorboard_runs/m1_record_spliter/ae-v0.1/run1')

In [ ]:
torch.cuda.empty_cache()

def col_fn(b): 
    return tuple(zip(*b))
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True,collate_fn=col_fn)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=True,collate_fn=col_fn)

mdl = BBModel().to(DEVICE)
mdl.backbone.encoder.requires_grad_(False)
opt = torch.optim.AdamW(mdl.parameters(), lr=0.0002)
sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,len(train_dl),T_mult=2)
crit = nn.MSELoss()

In [ ]:
# mdl.backbone.encoder.requires_grad_(True)

In [ ]:


# g_tloss = []
# g_vloss = []
# g_lr = []

for epoch in range(501):

    mdl.train()
    train_loss_avg = 0
    t = tqdm(train_dl,desc=f'Epoch {epoch}')
    for img,targ in t:
        img = [i.to(DEVICE) for i in img]
        targ = [{k:v.to(DEVICE) for k,v in i.items()} for i in targ]
    
        img,targ = aug(img,targ)
        targ = filter_rcnn_targets(targ)

        ret_dict = mdl(img,targ)

        loss = sum([v for k,v in ret_dict.items()])
        t.set_postfix_str(f'Loss: {loss.item():.03f}, Lr: {sch.get_last_lr()}')
        
        train_loss_avg += loss.item()

        loss.backward()
        opt.step()
        opt.zero_grad()

        sch.step()
    train_loss_avg/= len(train_dl)

    # LOSS ON VAL
    mdl.train()
    val_loss_avg = 0
    with torch.no_grad():
        for img,targ in val_dl:
            img = [i.to(DEVICE) for i in img]
            targ = [{k:v.to(DEVICE) for k,v in i.items()} for i in targ]

            # print(lab[:,:,0:4].size())
            ret_dict = mdl(img,targ)
            # loss = loss_classifier + loss_box_reg + loss_objectness + loss_rpn_box_reg
            val_loss_avg += sum([v for k,v in ret_dict.items()]).item()
    val_loss_avg /= len(val_dl)

    # PRED ON VAL
    mdl.eval()
    with torch.no_grad():
        d = mdl(img)

        img = [i.cpu() for i in img]
        d = [{k:v.cpu() for k,v in i.items()} for i in d]
        
        tb_writer.add_figure('Pred on val', show_bbox(img[0],d[0], targ[0], ret_fig=True,console=False),global_step=epoch,close=True)
    mdl.train()


    tb_writer.add_scalar('Train loss', train_loss_avg, epoch)
    tb_writer.add_scalar('Val loss', val_loss_avg, epoch)
    tb_writer.add_scalar('Learning rate', sch.get_last_lr()[0], epoch)


    # print(epoch+1, 'Train Loss:', train_loss_avg, 'Val Loss:',val_loss_avg)
    # plt.plot(g_tloss)
    # plt.plot(g_vloss)
    # plt.ylim(top=1,bottom=0)
    # plt.show()
    # plt.plot(g_lr)
    # plt.show()

In [ ]:
import pickle

In [ ]:
# pickle.dump(mdl, open('ae_run2_no-aug_12l.pkl','wb'))

# Active learning

In [ ]:
mdl = pickle.load(open('model_203e.pkl','rb'))

In [ ]:
from src.u1_downloader.images_loader import load_all
imgs_df = load_all('datasets/full_scans_random')

In [ ]:
def _uncertainty_score(pred):
    scores = pred["scores"]

    if len(scores) == 0:
        return 1.0

    top_scores = scores[:10]

    entropy = -(top_scores * torch.log(top_scores + 1e-8)).mean()

    return entropy.item()

In [ ]:
class ImagePathDS(Dataset):
    def __init__(self, image_paths):
        self.ip = image_paths
    def __getitem__(self,idx):
        return image_pre_trans(Image.open(self.ip[idx]))
    def __len__(self):
        return len(self.ip)

In [ ]:
def get_uncertainty_score_for_all(image_paths):
    ds = ImagePathDS(image_paths)

    uscore = []
    for img in tqdm(ds):
        img = img.unsqueeze(0)
        with torch.no_grad():
            img_cuda = img.to(DEVICE)

            d = mdl(img_cuda)

            # d = [{k:v.cpu() for k,v in i.items()} for i in d]
            # show_bbox(img,d)
        # print(uncertainty_score(d))
        for i in d:
            uscore.append(_uncertainty_score(i))
        # print(uscore[-1])
    return uscore

In [ ]:
u_scores = get_uncertainty_score_for_all(list(imgs_df.image_path))

In [ ]:


us = pd.Series(u_scores)
us[us==1] = 0

plt.plot(us)
plt.show()

print(f'Usable {(us<0.1).mean()*100:.02f}%')

K=len(us)//75
pooled = us.rolling(K).max().reset_index(drop=True)
us[pooled!=us] = 0
plt.plot(us)
plt.show()
choosen_to_label = imgs_df.image_path[us>0.1]
print('To label:', len(choosen_to_label))

In [ ]:
import shutil
for path in choosen_to_label:
    # print(r)
    print('Copy', path)
    shutil.copy(path, 'datasets/ds3_al' + '/' + path.split('/')[-1])
